# ContractNLI full dev: Qwen3-4B and Qwen3-1.7B, thinking off and on (Kaggle)

Runs the 5 scaffolds (`zeroshot`, `dspy`, `cot`, `rag`, `agentic_rag`) with thinking
**off** and **on** on the **full ContractNLI dev split** (61 contracts, 1,037 instances),
the same instances as the Qwen3-8B full-dev results. 2 models x 2 modes x 5 scaffolds = 20 runs.

**Settings (right sidebar):** Accelerator = **GPU T4 x2**, Internet = **On**.
Add the `contractnli-kaggle-full-dev` dataset via **+ Add Input**.

Best way to run: **Save Version -> Save & Run All (Commit)**. It runs in the background
(you can close the browser) and the results zip appears under the version's **Output**.
A commit is capped at 12 h. If it stops before finishing, add the previous version's
output as an input and run again: finished instances are skipped (step 2 restores them).

## 1. Install

In [ ]:
!pip install -q vllm "dspy==3.2.1" "litellm==1.73.1" sentence-transformers scikit-learn tiktoken requests
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Copy the files into the working directory

In [ ]:
import os, glob, shutil, zipfile

WORK = "/kaggle/working/contractnli"
os.makedirs(WORK, exist_ok=True)
NEEDED = ["dev.json", "run_matrix.py", "contractnli_zeroshot.py", "run_rlm_contractnli.py",
          "run_driver.py"]

# Kaggle unpacks an uploaded zip dataset into a folder; handle a nested zip too.
for f in NEEDED:
    hits = glob.glob(f"/kaggle/input/**/{f}", recursive=True)
    if hits:
        shutil.copy(hits[0], WORK)
for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
    with zipfile.ZipFile(z) as zf:
        for n in zf.namelist():
            if os.path.basename(n) in NEEDED and not os.path.exists(f"{WORK}/{os.path.basename(n)}"):
                with open(f"{WORK}/{os.path.basename(n)}", "wb") as out:
                    out.write(zf.read(n))

missing = [f for f in NEEDED if not os.path.exists(f"{WORK}/{f}")]
assert not missing, f"missing {missing} - did you add the dataset as input?"
os.environ["CONTRACTNLI_DATA_DIR"] = WORK

# Resume: if a previous version's output (results_qwen_full.zip) was added as input,
# restore its finished predictions so they are skipped.
for z in glob.glob("/kaggle/input/**/results_qwen_full.zip", recursive=True):
    with zipfile.ZipFile(z) as zf:
        zf.extractall(WORK)
    print("restored previous results from", z)
for d in glob.glob("/kaggle/input/**/results/matrix-*", recursive=True):
    shutil.copytree(d, f"{WORK}/results/{os.path.basename(d)}", dirs_exist_ok=True)
    print("restored", d)
print("files OK in", WORK)

## 3. Check this is the full dev split (1,037 instances)

In [ ]:
import sys
sys.path.insert(0, WORK)
from run_rlm_contractnli import load_split, iter_instances
n = len(list(iter_instances(load_split("dev"))))
print(n, "instances")
assert n == 1037, "dev.json is not the full ContractNLI dev split"

## 4. Serve both models (one per GPU) and run
Starts with a short smoke test (thinking really off / on), then the 20 runs.
Prints progress every 2 minutes. Finished instances are always skipped.

In [ ]:
MODELS = "Qwen/Qwen3-1.7B Qwen/Qwen3-4B"
!cd /kaggle/working/contractnli && python -u run_driver.py --models $MODELS

## 5. Summary tables

In [ ]:
M = "Qwen/Qwen3-4B Qwen/Qwen3-1.7B"
ARGS = "--report-only --n 1037 --models " + M
!cd /kaggle/working/contractnli && python run_matrix.py $ARGS
!cd /kaggle/working/contractnli && python run_matrix.py $ARGS --thinking

## 6. Package the results (download from the Output tab)

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/results_qwen_full", "zip",
                    "/kaggle/working/contractnli", "results")
shutil.make_archive("/kaggle/working/logs_qwen_full", "zip",
                    "/kaggle/working/contractnli", "logs")
print("-> /kaggle/working/results_qwen_full.zip")